# 00 — Data Download

Downloads both datasets to Google Drive for all downstream Colab notebooks.

| Dataset | GEO | Contents | Role |
|---------|-----|----------|------|
| Bhaduri et al. 2020, Nature | GSE132672 | Cortical organoids — 37 organoids, 3 protocols | Organoid data |
| Zhong et al. 2018, Nature | GSE104276 | Fetal PFC — gestational weeks 8–26 | Fetal brain reference |

**Run this notebook once.** After the data is on Drive, all other notebooks load from there.

## 1. Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Set Paths and Config

All paths are defined here. If you reorganize your Drive, only change this cell.

In [2]:
import os

DRIVE_ROOT = '/content/drive/MyDrive/brain-organoid-trajectories'

DATASETS = {
    'bhaduri_2020': {
        'accession': 'GSE132672',
        'description': 'Bhaduri et al. 2020 — cortical organoids (3 protocols)',
    },
    'zhong_2018': {
        'accession': 'GSE104276',
        'description': 'Zhong et al. 2018 — fetal PFC atlas, GW8-26',
    },
}

for name, info in DATASETS.items():
    info['raw_dir']       = os.path.join(DRIVE_ROOT, 'data', 'raw', name)
    info['processed_dir'] = os.path.join(DRIVE_ROOT, 'data', 'processed', name)
    os.makedirs(info['raw_dir'], exist_ok=True)
    os.makedirs(info['processed_dir'], exist_ok=True)
    print(f"[{info['accession']}] {info['description']}")
    print(f"  raw:       {info['raw_dir']}")
    print(f"  processed: {info['processed_dir']}\n")

[GSE132672] Bhaduri et al. 2020 — cortical organoids (3 protocols)
  raw:       /content/drive/MyDrive/brain-organoid-trajectories/data/raw/bhaduri_2020
  processed: /content/drive/MyDrive/brain-organoid-trajectories/data/processed/bhaduri_2020

[GSE104276] Zhong et al. 2018 — fetal PFC atlas, GW8-26
  raw:       /content/drive/MyDrive/brain-organoid-trajectories/data/raw/zhong_2018
  processed: /content/drive/MyDrive/brain-organoid-trajectories/data/processed/zhong_2018



## 3. Verify GEO Accessions — List Available Files

In [3]:
import ftplib

def list_geo_suppl(accession):
    prefix = accession[:len(accession)-3] + 'nnn'
    ftp_path = f'/geo/series/{prefix}/{accession}/suppl/'
    ftp = ftplib.FTP('ftp.ncbi.nlm.nih.gov')
    ftp.login()
    try:
        files = ftp.nlst(ftp_path)
        ftp.quit()
        return [f.split('/')[-1] for f in files]
    except ftplib.error_perm as e:
        ftp.quit()
        return [f'ERROR: {e}']

for name, info in DATASETS.items():
    acc = info['accession']
    print(f"--- {acc} ({info['description']}) ---")
    for f in list_geo_suppl(acc):
        print(f'  {f}')
    print()

--- GSE132672 (Bhaduri et al. 2020 — cortical organoids (3 protocols)) ---
  GSE132672_allorganoids_withnew_matrix.txt.gz

--- GSE104276 (Zhong et al. 2018 — fetal PFC atlas, GW8-26) ---
  GSE104276_all_pfc_2394_UMI_TPM_NOERCC.xls.gz
  GSE104276_all_pfc_2394_UMI_count_NOERCC.xls.gz
  GSE104276_readme_sample_barcode.xlsx
  filelist.txt
  GSE104276_RAW.tar



## 4. Download Both Datasets

Skips files that already exist (safe to re-run).

In [4]:
import ftplib
import os

def download_geo_suppl(accession, dest_dir):
    prefix = accession[:len(accession)-3] + 'nnn'
    ftp_path = f'/geo/series/{prefix}/{accession}/suppl/'
    ftp = ftplib.FTP('ftp.ncbi.nlm.nih.gov')
    ftp.login()
    files = ftp.nlst(ftp_path)
    print(f'  {len(files)} file(s) found')
    for filepath in files:
        filename = filepath.split('/')[-1]
        local_path = os.path.join(dest_dir, filename)
        if os.path.exists(local_path):
            size_mb = os.path.getsize(local_path) / 1e6
            print(f'  SKIP (exists, {size_mb:.1f} MB): {filename}')
            continue
        print(f'  Downloading: {filename} ...', end=' ', flush=True)
        with open(local_path, 'wb') as f:
            ftp.retrbinary(f'RETR {filepath}', f.write)
        size_mb = os.path.getsize(local_path) / 1e6
        print(f'done ({size_mb:.1f} MB)')
    ftp.quit()

for name, info in DATASETS.items():
    print(f"\n=== {info['accession']} — {name} ===")
    download_geo_suppl(info['accession'], info['raw_dir'])

print('\nAll downloads complete.')


=== GSE132672 — bhaduri_2020 ===
  1 file(s) found
  Downloading: GSE132672_allorganoids_withnew_matrix.txt.gz ... done (1578.0 MB)

=== GSE104276 — zhong_2018 ===
  5 file(s) found
  Downloading: GSE104276_all_pfc_2394_UMI_TPM_NOERCC.xls.gz ... done (33.2 MB)
  Downloading: GSE104276_all_pfc_2394_UMI_count_NOERCC.xls.gz ... done (16.0 MB)
  Downloading: GSE104276_readme_sample_barcode.xlsx ... done (0.3 MB)
  Downloading: filelist.txt ... done (0.0 MB)
  Downloading: GSE104276_RAW.tar ... done (38.7 MB)

All downloads complete.


## 5. Verify Downloads — List Files on Drive

In [5]:
for name, info in DATASETS.items():
    raw_dir = info['raw_dir']
    print(f"\n--- {name} ({raw_dir}) ---")
    total_mb = 0
    for fname in sorted(os.listdir(raw_dir)):
        fpath = os.path.join(raw_dir, fname)
        size_mb = os.path.getsize(fpath) / 1e6
        total_mb += size_mb
        print(f'  {fname:<60} {size_mb:>8.1f} MB')
    print(f'  Total: {total_mb:.1f} MB')


--- bhaduri_2020 (/content/drive/MyDrive/brain-organoid-trajectories/data/raw/bhaduri_2020) ---
  GSE132672_allorganoids_withnew_matrix.txt.gz                   1578.0 MB
  Total: 1578.0 MB

--- zhong_2018 (/content/drive/MyDrive/brain-organoid-trajectories/data/raw/zhong_2018) ---
  GSE104276_RAW.tar                                                38.7 MB
  GSE104276_all_pfc_2394_UMI_TPM_NOERCC.xls.gz                     33.2 MB
  GSE104276_all_pfc_2394_UMI_count_NOERCC.xls.gz                   16.0 MB
  GSE104276_readme_sample_barcode.xlsx                              0.3 MB
  filelist.txt                                                      0.0 MB
  Total: 88.2 MB


## 6. Install Dependencies

In [6]:
!pip install -q scanpy openpyxl xlrd

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 117.1 MB/s eta 0:00:00


## 7. Load Bhaduri 2020 (Organoids)

File: `GSE132672_allorganoids_withnew_matrix.txt.gz`  
Format: gzipped tab-separated text, genes × cells.

Uses a fast line-by-line sparse parser with `np.fromstring` (~10× faster than pandas chunks) and a 64 MB read buffer to minimize decompression overhead. Only non-zero values are stored. Takes ~5–10 minutes.

In [7]:
import io
import array
import numpy as np
import scipy.sparse as sp
import anndata
import scanpy as sc
import gzip

bhaduri_file = os.path.join(DATASETS['bhaduri_2020']['raw_dir'],
                             'GSE132672_allorganoids_withnew_matrix.txt.gz')

print('Reading Bhaduri 2020 — fast sparse parser...')

# array.array stores C primitives (4 bytes each) vs Python lists (28 bytes/object)
# ~7x memory reduction for data and col_indices
data        = array.array('f')  # float32
col_indices = array.array('i')  # int32
indptr      = [0]
gene_names  = []
cell_names  = None

with io.TextIOWrapper(io.BufferedReader(gzip.open(bhaduri_file, 'rb'), buffer_size=64 * 1024 * 1024)) as f:
    cell_names = f.readline().rstrip('\n').split('\t')

    for i, line in enumerate(f):
        line = line.rstrip('\n')
        tab = line.index('\t')
        gene_names.append(line[:tab])

        vals = np.fromstring(line[tab + 1:], dtype=np.float32, sep='\t')
        nz = np.flatnonzero(vals)
        data.extend(vals[nz].tolist())
        col_indices.extend(nz.astype(np.int32).tolist())
        indptr.append(indptr[-1] + len(nz))

        if (i + 1) % 2000 == 0:
            print(f'  {i + 1:,} genes processed...', flush=True)

print('Building sparse matrix...')
X = sp.csr_matrix(
    (np.frombuffer(data, dtype=np.float32),
     np.frombuffer(col_indices, dtype=np.int32),
     np.array(indptr, dtype=np.int32)),
    shape=(len(gene_names), len(cell_names))
).T.tocsr()  # transpose to cells x genes

adata_organoid = anndata.AnnData(X)
adata_organoid.obs_names = cell_names
adata_organoid.var_names = gene_names

print(f'\nDone. Shape: {adata_organoid.shape[0]:,} cells x {adata_organoid.shape[1]:,} genes')
print(f'Sparsity: {100 * (1 - X.nnz / (X.shape[0] * X.shape[1])):.1f}% zeros')
print(f'First 3 barcodes: {list(adata_organoid.obs_names[:3])}')
print(f'First 3 genes:    {list(adata_organoid.var_names[:3])}')

Reading Bhaduri 2020 — fast sparse parser...
  2,000 genes processed...
  4,000 genes processed...
  6,000 genes processed...
  8,000 genes processed...
  10,000 genes processed...
  12,000 genes processed...
  14,000 genes processed...
  16,000 genes processed...
Building sparse matrix...

Done. Shape: 242,350 cells x 16,774 genes
Sparsity: 92.2% zeros
First 3 barcodes: ['', 'H1SWeek3_AAACCTGAGACAAAGG', 'H1SWeek3_AAACCTGAGCACACAG']
First 3 genes:    ['FO538757.2', 'AP006222.2', 'RP11-206L10.9']


## 8. Load Zhong 2018 (Fetal PFC)

File: `GSE104276_all_pfc_2394_UMI_count_NOERCC.xls.gz`  
We use raw UMI counts (not TPM) so we can apply our own normalization.  
Despite the `.xls` extension, GEO files like this are typically tab-separated text.

In [9]:
import pandas as pd

zhong_file = os.path.join(DATASETS['zhong_2018']['raw_dir'],
                           'GSE104276_all_pfc_2394_UMI_count_NOERCC.xls.gz')

print('Loading Zhong 2018...')

# GEO .xls files are usually tab-separated text despite the extension
try:
    df_fetal = pd.read_csv(zhong_file, sep='\t', index_col=0, compression='gzip')
    print('Loaded as tab-separated text.')
except Exception:
    # Fallback: try as actual Excel
    df_fetal = pd.read_excel(zhong_file, index_col=0)
    print('Loaded as Excel.')

print(f'Raw matrix shape: {df_fetal.shape[0]:,} genes x {df_fetal.shape[1]:,} cells')

# Transpose: scanpy expects cells x genes
adata_fetal = anndata.AnnData(df_fetal.T)
print(f'AnnData shape:    {adata_fetal.shape[0]:,} cells x {adata_fetal.shape[1]:,} genes')
print(f'First 3 barcodes: {list(adata_fetal.obs_names[:3])}')
print(f'First 3 genes:    {list(adata_fetal.var_names[:3])}')

Loading Zhong 2018...
Loaded as tab-separated text.
Raw matrix shape: 24,153 genes x 2,394 cells
AnnData shape:    2,394 cells x 24,153 genes
First 3 barcodes: ['GW08_PFC1_sc1', 'GW08_PFC1_sc2', 'GW08_PFC1_sc3']
First 3 genes:    ['A1BG', 'A1BG-AS1', 'A1CF']


## 9. Save as h5ad

Convert to `.h5ad` — scanpy's native format, fast to load in all downstream notebooks.  
No filtering or normalization applied — this is the raw object.

In [10]:
def save_h5ad(adata, processed_dir, filename):
    out_path = os.path.join(processed_dir, filename)
    adata.write_h5ad(out_path)
    size_mb = os.path.getsize(out_path) / 1e6
    print(f'Saved: {out_path} ({size_mb:.1f} MB)')

save_h5ad(adata_organoid, DATASETS['bhaduri_2020']['processed_dir'], 'bhaduri_2020_raw.h5ad')
save_h5ad(adata_fetal,    DATASETS['zhong_2018']['processed_dir'],   'zhong_2018_raw.h5ad')

Saved: /content/drive/MyDrive/brain-organoid-trajectories/data/processed/bhaduri_2020/bhaduri_2020_raw.h5ad (2559.6 MB)
Saved: /content/drive/MyDrive/brain-organoid-trajectories/data/processed/zhong_2018/zhong_2018_raw.h5ad (463.7 MB)


## Done

Data is now on Google Drive:
- `data/raw/bhaduri_2020/` — original GEO files
- `data/raw/zhong_2018/` — original GEO files
- `data/processed/bhaduri_2020/bhaduri_2020_raw.h5ad`
- `data/processed/zhong_2018/zhong_2018_raw.h5ad`

Next: `colab_01_preprocessing.ipynb` — QC, filtering, normalization, HVG, PCA on both datasets.